# Nesterov Accelerated Gradient (NAG)

Nesterov Accelerated Gradient (NAG) is a smart modification of SGD with Momentum. While standard momentum blindly rolls down the hill, NAG looks ahead at where it is going and applies the brakes if it is about to overshoot.


## 1. The Core Intuition: The Driver with Eyes vs. The Blind Bowling Ball

* **Standard Momentum** is like a **blind bowling ball** rolling down a valley. It accumulates speed and rolls right past the bottom (minimum point) because it doesn't realize the slope has changed until it has already crossed it. It is forced to overshoot and wobble back and forth.
* **Nesterov Momentum (NAG)** is like a **smart driver** looking through the windshield. The driver knows the car's current speed and direction (momentum). Before making the next move, the driver projects where the car will be in the next second. If they see a sharp turn or an uphill climb coming up, they tap the brakes **before** reaching it to prevent overshooting.


## 2. The Mathematics: Looking Ahead

NAG calculates the gradient (slope) not at the current position, but at the **predicted next position** (where our momentum is about to take us).

### **Nesterov Accelerated Gradient (NAG) Equations**
First, we look ahead by applying our momentum to our current position: $w_t - \beta \cdot v_{t-1}$. Then, we compute the gradient at that look-ahead point:

1. **Calculate Velocity (using look-ahead gradient):**
   $$v_t = \beta \cdot v_{t-1} + \eta \cdot \nabla L(w_t - \beta \cdot v_{t-1})$$
2. **Update the Weights:**
   $$w_{t+1} = w_t - v_t$$

### **Symbol Legend:**
* **$w_t$:** Current weight positions.
* **$w_t - \beta \cdot v_{t-1}$:** The **look-ahead position** (approximate next position based purely on past velocity).
* **$\nabla L(\cdot)$:** The gradient (slope) calculated at the specified position.
* **$v_t$:** The updated velocity vector.
* **$\beta$:** Momentum coefficient (friction, usually `0.9`).
* **$\eta$:** Learning rate.
* **$\nabla L(w_t - \beta \cdot v_{t-1})$** : This simply means gradient at this point
   * **$w_t$**: Your **current position** (current weight values).
   * **$\beta \cdot v_{t-1}$**: Your **momentum step** (the velocity from the previous step, scaled by $\beta$). This is the direction and distance you know you are about to travel due to momentum.
   * **$w_t - \beta \cdot v_{t-1}$**: The **look-ahead position**. It is an estimation of where your weights will land in the next step.
   * **$\nabla L(\dots)$**: The **gradient (slope)**. It tells you which direction is "uphill" and how steep it is.
   Putting it all together, $\nabla L(w_t - \beta \cdot v_{t-1})$ implies:
   > **"Calculate the slope of the loss function not at our current position ($w_t$), but at the predicted future position ($w_t - \beta \cdot v_{t-1}$)."**


> **SO BASCIALLY THE FIRST PART OF THE FORMULA TELLS WHAT WILL BE THE VELOCITY AND THE LATER PART OF THE FORMULAS TELLS WHERE THE BALL SHOULD MOVE.**
1. **The First Equation (Velocity Calculation):**
   * **What it does:** Calculates the **new velocity** (speed and direction) of the ball. It blends the previous velocity history ($\gamma v_{t-1}$) with the look-ahead gradient force.
2. **The Second Equation (Movement Step):**
   * **What it does:** Dictates **where the ball actually moves**. It updates the current parameters ($\theta_t$) by subtracting the newly calculated velocity ($v_t$) to find the next position.


## 3. Visual Comparison of the Steps

Here is how the update steps differ geometrically:

### **Standard Momentum**
```text
           [Momentum Step (β * v_t-1)]
Current Pos ============================> Temp Point
                                              \
                                               \  [Current Gradient Step]
                                                v
                                            New Position
```
*Standard momentum makes a big leap forward based on history, and only then calculates the local gradient at the starting position to adjust.*

### **Nesterov Momentum (NAG)**
```text
           [Momentum Step (β * v_t-1)]
Current Pos ============================> Look-Ahead Point
                                              /
                                             /  [Look-Ahead Gradient Step]
                                            v
                                      New Position
```
*NAG projects forward to the look-ahead point first, calculates the gradient at that future location, and uses that future gradient to drag the final position back toward the safety zone.*


## 4. Why We Use It: The "Braking" Effect
Imagine the optimizer is heading fast toward the bottom of a valley. 
* Under **Standard Momentum**, the current gradient is still pointing downward, so it adds more speed, causing a massive overshoot.
* Under **NAG**, the look-ahead point lands on the opposite upward slope. The gradient calculated at this future point points **backward** (positive gradient trying to push the weights back). This backward force is added to the velocity, acting as an **automatic brake** that slows the optimizer down just before it hits the bottom.


## Pros (Advantages)
- **Reduced Overshooting:** The look-ahead mechanism acts as a smart brake, slowing down the optimizer before it flies too far past the minimum.
- **Faster Convergence & Stability:** By suppressing unnecessary oscillations around the minimum, it converges to the valley bottom much quicker than standard momentum.
- **Smarter Path Correction:** It dynamically adjusts its trajectory based on the slope of the upcoming terrain, leading to a more responsive optimization path.
## Cons (Disadvantages)
- **Prone to Traps (More Conservative):** Because it actively brakes when encountering uphill slopes, it is actually more likely to settle and get trapped in local minima rather than blindly flying out of them.
- **No Adaptive Learning Rate:** It still applies a single global learning rate to all parameters, meaning it does not handle sparse features or parameters requiring different step sizes well on its own.
- **Extra Hyperparameter:** Requires manual tuning of the momentum parameter, just like standard momentum.
- **Implementation Complexity:** Evaluating gradients at a shifted look-ahead position is mathematically more complex, though modern frameworks (like PyTorch) handle this under the hood using an approximation.


## 6. Keras Code Implementation

In Keras, you do not need to write the look-ahead math yourself. You simply use the standard `SGD` optimizer and set the `nesterov` flag to `True`:

```python
from tensorflow import keras

# Define SGD with Nesterov Momentum
opt = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True)

# Compile the model
model.compile(optimizer=opt, loss='mean_squared_error')
```